In [ ]:
!pip install -q marlin-pytorch opencv-python-headless scikit-learn pandas tqdm matplotlib

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Пути — замени название датасета на своё
input_dir = Path("/kaggle/input/datasets/katushkastokom/marlin-face-crops")
face_crops_dir = input_dir / "face_crops-20260411T211821Z-3-001/face_crops"
subject_map_path = input_dir / "subject_map.csv"

work_dir = Path("/kaggle/working")
emb_dir = work_dir / "embeddings"
(emb_dir / "truthful").mkdir(parents=True, exist_ok=True)
(emb_dir / "deceptive").mkdir(parents=True, exist_ok=True)

random_seed = 42
test_size = 0.2

# Проверка
print("face_crops найден:", face_crops_dir.exists())
print("subject_map найден:", subject_map_path.exists())
video_exts = {".mp4", ".mov", ".avi", ".mkv"}
all_face_videos = sorted([
    p for p in face_crops_dir.rglob("*")
    if p.suffix.lower() in video_exts
])
print("Видео найдено:", len(all_face_videos))

In [ ]:
rows = []
for vp in all_face_videos:
    label_name = "deceptive" if "_lie_" in vp.name else "truthful"
    label = 0 if label_name == "deceptive" else 1
    subject_id = vp.stem.split("_")[0]
    rows.append({
        "video": vp.name,
        "face_video_path": str(vp),
        "label_name": label_name,
        "label": label,
        "subject_id": subject_id,
    })

meta_df = pd.DataFrame(rows)
print("meta_df готов:", len(meta_df), "видео")
print(meta_df["label_name"].value_counts())
print("Уникальных людей:", meta_df["subject_id"].nunique())

In [ ]:
import torch
from marlin_pytorch import Marlin

print("Загружаем MARLIN vit_base...")
model = Marlin.from_online("marlin_vit_base_ytf")
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Устройство:", device)
print("Модель загружена")

In [ ]:
from tqdm import tqdm

records = []

for _, row in tqdm(meta_df.iterrows(), total=len(meta_df), desc="MARLIN vit_base"):
    label_name = row["label_name"]
    vp = Path(row["face_video_path"])
    out_path = emb_dir / label_name / f"{vp.stem}.npy"

    if out_path.exists():
        existing = np.load(out_path)
        records.append({
            **row.to_dict(),
            "embedding_path": str(out_path),
            "dim": int(existing.shape[0]),
            "error": None,
        })
        continue

    try:
        with torch.no_grad():
            feats = model.extract_video(str(vp))
            mean_emb = feats.mean(dim=0)
            std_emb = feats.std(dim=0)
            emb = torch.cat([mean_emb, std_emb], dim=0).detach().cpu().numpy()
        np.save(out_path, emb)
        records.append({
            **row.to_dict(),
            "embedding_path": str(out_path),
            "dim": int(emb.shape[0]),
            "error": None,
        })
    except Exception as e:
        records.append({
            **row.to_dict(),
            "embedding_path": None,
            "dim": None,
            "error": str(e),
        })

emb_df = pd.DataFrame(records)
emb_df.to_csv(work_dir / "embeddings_index_base.csv", index=False)
print("Готово:", emb_df["embedding_path"].notna().sum(), "из", len(emb_df))
print("Размерность:", emb_df["dim"].iloc[0])

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, roc_auc_score, matthews_corrcoef)

emb_df = pd.read_csv(work_dir / "embeddings_index_base.csv")
emb_df = emb_df[emb_df["embedding_path"].notna()].copy()

X = np.stack([np.load(p) for p in emb_df["embedding_path"]])
y = emb_df["label"].astype(int).to_numpy()
groups = emb_df["subject_id"].astype(str).to_numpy()

print("X shape:", X.shape)
print("Уникальных людей:", len(set(groups)))

classifiers = {
    "LogisticRegression": LogisticRegression(max_iter=5000, class_weight="balanced"),
    "SVM_RBF": SVC(kernel="rbf", class_weight="balanced", probability=True),
    "SVM_Linear": SVC(kernel="linear", class_weight="balanced", probability=True),
    "RandomForest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=random_seed),
}

results = []
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=random_seed)

for clf_name, clf_model in classifiers.items():
    fold_metrics = []
    for fold, (tr_idx, te_idx) in enumerate(sgkf.split(X, y, groups=groups), start=1):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("model", clf_model)
        ])
        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)
        y_prob = pipe.predict_proba(X_te)[:, 1]

        fold_metrics.append({
            "accuracy": accuracy_score(y_te, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_te, y_pred),
            "f1": f1_score(y_te, y_pred, average="macro"),
            "auc": roc_auc_score(y_te, y_prob),
            "mcc": matthews_corrcoef(y_te, y_pred),
        })

    fm = pd.DataFrame(fold_metrics)
    results.append({
        "classifier": clf_name,
        "accuracy": round(fm["accuracy"].mean(), 4),
        "balanced_accuracy": round(fm["balanced_accuracy"].mean(), 4),
        "f1_macro": round(fm["f1"].mean(), 4),
        "auc": round(fm["auc"].mean(), 4),
        "mcc": round(fm["mcc"].mean(), 4),
        "std_accuracy": round(fm["accuracy"].std(), 4),
    })

results_df = pd.DataFrame(results).sort_values("balanced_accuracy", ascending=False)
results_df.to_csv(work_dir / "classifier_comparison_base.csv", index=False)
print("\nРезультаты:")
print(results_df.to_string(index=False))